# 07 — Docker, containers & Azure Container Apps

**Objectif :** comprendre les **containers** (Docker), les **registres d'images** (ACR) et la plateforme **Azure Container Apps** (ACA) qui orchestre tout ça. À la fin, on **build et déploie un mini container Hello World** sur ACA.

C'est l'avant-dernière brique avant d'attaquer le projet `mailroom-ia` qui utilise précisément ACA Environment + Apps + Jobs.

À l'issue, vous saurez :
- ce qu'est un **container** vs une VM, et pourquoi c'est devenu le standard ;
- lire et écrire un **Dockerfile** simple (multi-stage) ;
- ce qu'est une **image**, un **registry** (Docker Hub, **Azure Container Registry**) ;
- la différence **Container App** vs **Container App Job** (Apps & Jobs) ;
- ce qu'est **KEDA** et comment ça scale event-driven ;
- déployer concrètement un container sur ACA en quelques commandes.

> 🔜 Le **notebook 08** parle des **bases de données et du stockage Azure**, puis le **notebook 09** enchaîne sur **Bicep / Infrastructure as Code** et contient le **cleanup final** de tout le parcours (notebook 09).

> ⚠️ Ce notebook **utilise Docker en local** pour la partie hands-on (sections 4 et 8). Si vous êtes en Codespaces : Docker est déjà dispo. Si vous êtes en local Windows : il faut **Docker Desktop** installé.

## 1. Container vs VM — la différence en 1 schéma

```
  VIRTUAL MACHINE                          CONTAINER
  ───────────────                          ─────────
  ┌─────────────────────────┐              ┌─────────────────────────┐
  │  App                    │              │  App                    │
  ├─────────────────────────┤              ├─────────────────────────┤
  │  Libs / runtime         │              │  Libs / runtime         │
  ├─────────────────────────┤              ├─────────────────────────┤
  │  OS guest (Linux/Windows)│              │ (rien — partage du noyau)│
  ├─────────────────────────┤              ├─────────────────────────┤
  │  Hyperviseur            │              │  Container runtime      │
  ├─────────────────────────┤              ├─────────────────────────┤
  │  OS hôte                │              │  OS hôte (noyau partagé)│
  ├─────────────────────────┤              ├─────────────────────────┤
  │  Hardware               │              │  Hardware               │
  └─────────────────────────┘              └─────────────────────────┘
       lourd, ~GB                              léger, ~MB
       démarre en min                          démarre en sec
       isolation totale                        isolation processus
```

Un **container** = un processus isolé (cgroups + namespaces Linux) qui embarque son code + ses libs, **sans OS complet**. Il partage le noyau de l'hôte.

Conséquences :
- **Démarre en quelques secondes** (vs minutes pour une VM)
- **Image ~50-300 MB** (vs plusieurs GB pour une VM)
- **Densité** : 50 containers/VM sans souci
- **Portable** : la même image tourne sur votre PC, en CI, en prod
- **Immuable** : on ne *modifie* pas un container, on en *recrée* un autre

## 2. Vocabulaire à connaître

| Terme | Définition |
|-------|------------|
| **Image** | Le « modèle figé » : code + libs + config. Read-only. Identifiée par `nom:tag` (ex. `python:3.13-slim`). |
| **Container** | Une **instance** d'une image qui tourne. On peut en lancer N depuis la même image. |
| **Dockerfile** | Recette texte pour fabriquer une image, étape par étape (cf. §3). |
| **Layer** | Une étape Dockerfile = une couche cacheable. Les images partagent leurs couches communes. |
| **Registry** | Un dépôt d'images. Public (**Docker Hub**, `mcr.microsoft.com`) ou privé (**Azure Container Registry**, GitHub Container Registry…). |
| **Tag** | Un label sur une image (`v1`, `latest`, `2.5.0`). **`latest` ne veut rien dire** — toujours pinner. |
| **Digest** | Hash SHA-256 immuable de l'image (`sha256:abcd...`). C'est le vrai identifiant. |
| **Engine / runtime** | Le programme qui exécute les containers : Docker, containerd, Podman… |
| **OCI** | Le standard ouvert (Open Container Initiative) qui définit le format image + runtime. Toutes les implémentations sont compatibles. |

## 3. Microservices vs monolithe — pourquoi on découpe

Avant de parler d'orchestration, il faut comprendre **pourquoi tu te retrouves avec plusieurs containers** au lieu d'une seule grosse application.

### 3.1 L'app monolithique — l'approche historique

Imagine un site e-commerce **monolithique** :

```
┌─────────────────────────────────────────────────────┐
│  Mon-Super-Shop  (1 seul gros process, 1 seul code) │
├─────────────────────────────────────────────────────┤
│  • Frontend (HTML, JS rendus côté serveur)          │
│  • API REST (login, panier, commandes)              │
│  • Logique paiement                                 │
│  • Logique stock                                    │
│  • Notifications email                              │
│  • Génération de PDF (factures)                     │
│  • Recherche produit                                │
│  • Recommandations IA                               │
└─────────────────────────────────────────────────────┘
        │
        ▼
   PostgreSQL (tout dans une seule base)
```

**Avantages d'un monolithe** :
- Simple à développer au début, simple à déployer (1 build, 1 release).
- Pas de réseau interne à gérer (tout est en mémoire dans le même process).
- Transactions DB triviales (1 seule DB).

**Inconvénients quand ça grossit** :
- ⏱ Un bug dans la **recherche** crash tout le site, y compris le panier.
- 🐌 Pour passer à l'échelle, tu dois cloner **toute** l'app — alors qu'il n'y a peut-être que la recherche qui rame.
- 🧱 Un développeur Python qui touche les recommandations IA peut casser sans le savoir le module facturation .NET.
- 🚀 **1 déploiement par jour max** — tester un monolithe complet prend des heures.
- 🤝 Les équipes se marchent dessus dans le même repo.

### 3.2 La même app en microservices

```
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
│ frontend │   │  panier  │   │ paiement │   │  search  │
│ (Next.js)│   │  (Node)  │   │  (Java)  │   │ (Python) │
└──────────┘   └──────────┘   └──────────┘   └──────────┘
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
│  stock   │   │ emails   │   │ factures │   │  reco-IA │
│   (Go)   │   │  (Node)  │   │ (Python) │   │ (Python) │
└──────────┘   └──────────┘   └──────────┘   └──────────┘
       │             │              │              │
       └─── chaque service a sa propre DB ────────┘
   PostgreSQL · Redis · MongoDB · ... selon les besoins
```

Chaque service est :
- **Indépendant** : son propre repo (ou dossier), sa propre stack, sa propre DB.
- **Petit** : 1 service = 1 responsabilité (Single Responsibility Principle au niveau système).
- **Déployable séparément** : une équipe peut releaser `paiement` 10 fois par jour sans toucher `panier`.
- **Scalable séparément** : seul `search` rame ? On scale `search` à 20 replicas, les autres restent à 1.

**Le prix à payer** :
- 🔌 Les services se parlent par **réseau** (HTTP/gRPC/queue), donc latence + erreurs réseau.
- 🐛 Un bug "le panier ne calcule pas la TVA" peut maintenant impliquer 5 services → debugging plus dur, **tracing distribué nécessaire** (App Insights, OpenTelemetry).
- 💾 Pas de transaction DB cross-service — il faut des patterns comme **Saga**, **event sourcing**, **outbox**.
- 🛠️ Plus de plomberie : service discovery, secrets, observabilité, identités…
- 🐳 **Beaucoup, beaucoup, beaucoup de containers à orchestrer.**

### 3.3 Notre projet `mailroom-ia` — 2 microservices, pas 1 monolithe

On aurait pu coder `mailroom-ia` en monolithe : 1 seule app Next.js qui upload le PDF, appelle Document Intelligence et le LLM Foundry, écrit dans Cosmos, le tout dans la même requête HTTP.

Pourquoi on l'a **séparé en 2 services** ?

| Service | Rôle | Pourquoi séparé |
|---------|------|-----------------|
| **`web`** (Next.js) | Frontend admin + API upload (HTTP synchrone, < 1 s) | Doit répondre vite, scale sur charge HTTP |
| **`worker`** (Python) | OCR + LLM + Cosmos (asynchrone, ~25 s par doc) | Travail lourd, peut tourner en parallèle, **scale-to-zero** quand pas de doc |

Si on avait gardé un monolithe :
- ❌ L'upload bloquerait 25 secondes (le navigateur timeout, l'admin pète un câble).
- ❌ On payerait du compute Node.js 24/7 juste pour faire tourner du Python OCR épisodique.
- ❌ Une exception dans le worker crasherait potentiellement la page admin.

Avec 2 services :
- ✅ L'upload répond en 200 ms (juste un PUT blob + enqueue).
- ✅ Le worker tourne **seulement quand y'a une queue** (KEDA scale-to-zero).
- ✅ Web et worker ont des **stacks différentes** (TypeScript vs Python) — chacun avec son SDK adapté.

**Règle d'or** : sépare en microservices quand tu as une **bonne raison** (perf, équipe, scaling, techno) — pas par mode. 2-5 services c'est sain. 50 services pour un POC = catastrophe.


## 4. Le problème — quand tu as plusieurs containers en prod

OK, donc tu as N containers à faire tourner. Naïvement, sur un serveur Linux, tu pourrais faire :

```bash
docker run -d --name web myorg/web:v1
docker run -d --name worker myorg/worker:v1
docker run -d --name redis redis:7
```

**Ça marche… pendant 10 minutes.** Puis les problèmes arrivent.

### 4.1 Les questions que tu te poses très vite

| # | Question | Sans orchestrateur |
|---|----------|---------------------|
| 1 | Le container `worker` a crashé à 3h du mat'. Qu'est-ce qui le redémarre ? | Toi. Réveil. |
| 2 | Je veux 5 replicas de `web` pour absorber la charge. Comment ? | `docker run` 5 fois, gérer les ports à la main. |
| 3 | Comment je load-balance les requêtes HTTP entre les 5 replicas ? | Installer nginx, le configurer à la main, le reconfigurer à chaque scale. |
| 4 | Mon serveur tombe en panne. Que se passe-t-il ? | Tout est down jusqu'à ce qu'un humain bouge. |
| 5 | Je déploie `web:v2`. Comment je passe de v1 à v2 sans downtime ? | Script bash custom, prier. |
| 6 | Et si v2 est buggée, comment je rollback en 30 secondes ? | Re-déploiement complet, prier plus fort. |
| 7 | Comment `web` trouve l'IP de `worker` ? | `/etc/hosts` à la main, refresh manuel quand worker bouge. |
| 8 | Comment je sais que `worker` consomme 95% de CPU ? | Tu ne le sais pas avant qu'un client se plaigne. |
| 9 | Comment je partage un secret entre 3 containers sans le mettre en clair ? | Variables d'env… qui leakent dans les logs. |
| 10 | Comment je passe à 2 serveurs ? 10 ? 100 ? | Tu réécris tout ton tooling. Bonne chance. |

### 4.2 Le pattern qu'on veut

Pour résoudre tout ça, on veut un **orchestrateur de containers** : un système qui prend en charge :

- **Scheduling** : "place ce container sur la machine la moins chargée".
- **Self-healing** : "si un container meurt, relance-le tout seul".
- **Scaling horizontal** : "passe de 3 à 10 replicas quand le CPU dépasse 70%".
- **Service discovery** : "ce container peut joindre `worker` à l'URL `http://worker:8080`, peu importe où `worker` tourne".
- **Load balancing** : "distribue le trafic entre tous les replicas sains".
- **Rolling updates** : "passe de v1 à v2 progressivement, garde le service up".
- **Secrets & config** : "injecte les secrets de manière sécurisée, sans les leak".
- **Storage** : "monte un volume persistant si le container a besoin de stocker".
- **Observability** : "centralise les logs et métriques de tous les containers".
- **Network policies** : "le service `paiement` ne peut PAS joindre `frontend`".

C'est ce que fait un **orchestrateur de containers**. Et le standard de facto pour ça, c'est **Kubernetes**.


## 5. Kubernetes (k8s) — le standard de l'orchestration

Kubernetes est un projet open-source créé par Google en 2014, basé sur Borg (leur orchestrateur interne depuis 2003). Aujourd'hui c'est **LE** standard de l'orchestration de containers — utilisé par Google, Spotify, Netflix, Airbnb, et 90% des grandes entreprises.

### 5.1 Comment k8s résout les problèmes ci-dessus

| Problème | Solution Kubernetes |
|----------|---------------------|
| Self-healing | **kubelet** surveille chaque pod, le redémarre s'il crashe |
| Scaling | **Horizontal Pod Autoscaler** ajuste les replicas sur CPU/RAM/metric custom |
| Service discovery | Chaque **Service** k8s = une IP virtuelle stable + un DNS interne |
| Load balancing | Inclus dans l'objet **Service** (round-robin par défaut) |
| Rolling updates | Objet **Deployment** = montée progressive + readiness probe |
| Secrets | Objet **Secret** = stocké chiffré, monté en file ou env var |
| Network policies | Objet **NetworkPolicy** = firewall L4 entre pods |
| Observability | Tout container envoie ses logs au stdout → kubelet collecte |

### 5.2 Vocabulaire k8s minimum (5 mots à connaître)

Tu n'as pas besoin de devenir expert k8s, mais ces 5 mots fuient partout dans les docs Azure :

| Mot | Définition pour un débutant |
|-----|------------------------------|
| **Cluster** | Un groupe de machines (= **nodes**) qui exécutent tes containers ensemble |
| **Node** | Une machine du cluster (typiquement une VM Linux) |
| **Pod** | Une unité de déploiement = 1 ou plusieurs containers qui partagent réseau/disque. **La plupart du temps : 1 pod = 1 container.** |
| **Deployment** | Une définition qui dit "je veux N pods de telle image, avec telle config" |
| **Service** | Une IP virtuelle + DNS interne qui load-balance vers les pods |

Visuellement :
```
Cluster Kubernetes
├── Node 1 (VM)
│   ├── Pod A  (web replica 1)
│   ├── Pod B  (web replica 2)
│   └── Pod C  (redis)
├── Node 2 (VM)
│   ├── Pod D  (web replica 3)
│   └── Pod E  (worker)
└── Node 3 (VM)
    └── Pod F  (worker)

Deployment "web": je veux 3 replicas  → Pods A, B, D
Service "web":    IP virtuelle 10.0.0.42  → load-balance vers A, B, D
```

### 5.3 Pourquoi Kubernetes peut être trop lourd

Kubernetes résout tous les problèmes ci-dessus… mais introduit ses propres complexités :

| Tu dois apprendre | Pourquoi c'est dur |
|-------------------|---------------------|
| **YAML** (énormément) | 200+ champs, indentation sensible |
| **RBAC k8s** | Système séparé d'Azure RBAC, avec son propre vocabulaire |
| **Networking k8s** | CNI, ingress controllers, network policies, services |
| **Helm / Kustomize** | Templater le YAML pour multi-env |
| **Patches & upgrades** | Versions k8s tous les 3 mois, breaking changes possibles |
| **Operators & CRDs** | Étendre k8s avec des opérateurs custom |
| **Sécurité** | Pod Security Standards, secrets, sealed-secrets, vault |

**En vrai, pour faire tourner k8s en prod sereinement il te faut 1 à 3 ingénieurs DevOps à plein temps.**

C'est rentable quand tu as 50+ microservices, ou une équipe de plateforme dédiée. Pour un POC à 2 containers comme `mailroom-ia` ? C'est un marteau de forgeron pour planter un clou.

### 5.4 La conclusion — on a besoin d'un orchestrateur, mais pas forcément de k8s nu

On veut **les bénéfices** de k8s (self-healing, scaling, service discovery…) **sans** :
- Apprendre 200 champs YAML.
- Gérer le control plane k8s.
- Maintenir 3 ingénieurs DevOps.

**C'est exactement ce que propose Azure Container Apps.**


## 6. Anatomie d'un Dockerfile

Voici le **Dockerfile multi-stage** du worker du projet `mailroom-ia` (simplifié) :

```dockerfile
# syntax=docker/dockerfile:1.7

# ─── STAGE 1 : builder ───
FROM python:3.13-slim AS builder
WORKDIR /build
RUN pip install --no-cache-dir uv
COPY pyproject.toml ./
RUN uv pip compile pyproject.toml -o requirements.txt

# ─── STAGE 2 : runtime ───
FROM python:3.13-slim
WORKDIR /app

# Bonnes pratiques Python conteneurisé
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1

# Utilisateur non-root (sécurité)
RUN groupadd -r app && useradd -r -g app -d /app app

# Copie depuis le stage builder
COPY --from=builder /build/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY worker/ ./worker/
USER app

CMD ["python", "-m", "worker.main"]
```

**Pourquoi multi-stage ?**
- Le `builder` peut contenir des outils lourds (compilateurs, dev deps) ;
- L'image finale ne garde que **le strict nécessaire** au runtime.
- Résultat : image **2-5× plus petite**, surface d'attaque réduite.

**Bonnes pratiques (à mémoriser)** :

| Règle | Pourquoi |
|-------|----------|
| Image de base **`-slim`** ou **`-alpine`** | Plus petite, moins de CVE |
| Ne **pas tourner en root** (`USER app`) | Sécurité — moindre privilège |
| **Pin** les versions (`python:3.13-slim`, pas `python:latest`) | Reproductibilité |
| `.dockerignore` propre (exclure `.git`, `node_modules`, etc.) | Build plus rapide, image plus petite |
| Ordre des `COPY` : du plus stable au plus changeant | Optimise le cache des couches |
| **Healthcheck** côté Dockerfile ou côté orchestrateur (ACA) | Détection de pannes |

## 7. Hands-on local — build, run, inspect

On va builder un mini Hello-World en Python directement dans une cellule. On crée le Dockerfile à la volée.

In [ ]:
# Crée un répertoire de travail temporaire pour le mini container
from pathlib import Path
DEMO = Path("./_docker-demo")
DEMO.mkdir(exist_ok=True)

(DEMO / "app.py").write_text(
    "import os, sys\n"
    "print(f'Hello depuis {sys.platform} Python {sys.version.split()[0]}')\n"
    "print(f'Hostname container : {os.uname().nodename}')\n",
    encoding="utf-8",
)

(DEMO / "Dockerfile").write_text(
    "FROM python:3.13-slim\n"
    "WORKDIR /app\n"
    "COPY app.py .\n"
    'CMD ["python", "app.py"]\n',
    encoding="utf-8",
)

print("✓ Dockerfile + app.py créés dans", DEMO.resolve())

In [ ]:
# Vérifier que Docker tourne
!docker version

In [ ]:
# Build l'image (≈ 30 sec la première fois)
!docker build -t hello-stage:v1 ./_docker-demo

In [ ]:
# Lister mes images locales
!docker images hello-stage

In [ ]:
# Lancer le container (instance de l'image)
!docker run --rm hello-stage:v1

In [ ]:
# Inspecter les couches (layers) de l'image
!docker history hello-stage:v1

## 8. Les registres — où vivent les images

Une image construite localement n'est dispo que sur **votre machine**. Pour qu'un cluster Azure puisse la déployer, il faut la **pousser dans un registre**.

| Registre | Usage |
|----------|-------|
| **Docker Hub** | Public par défaut. Beaucoup d'images de base (`python`, `node`…). Limites pull pour utilisateurs anonymes. |
| **`mcr.microsoft.com`** | Microsoft Container Registry — images Microsoft (devcontainers, .NET, SQL Server…). Public, sans limite. |
| **GitHub Container Registry** (`ghcr.io`) | Lié à GitHub Actions, gratuit, populaire en CI/CD. |
| **Azure Container Registry (ACR)** ⭐ | Registry privé Azure, intégré RBAC + Managed Identity. **C'est ce qu'on utilise pour le projet `mailroom-ia`.** |

### Le flow standard

```
  docker build → image locale
        │
        ▼
  docker tag <local> <registry>/<repo>:<tag>
        │
        ▼
  docker login <registry>     ← ou MI / OIDC en CI
        │
        ▼
  docker push <registry>/<repo>:<tag>
        │
        ▼
  L'image est dispo dans le registry → ACA / AKS / autre orchestrateur peut la pull
```

### `az acr build` — build cloud-side

Pour éviter d'avoir Docker installé localement, **Azure peut builder pour vous** :

```bash
az acr build --registry <acr-name> --image hello:v1 .
```

→ ACR fait le `docker build` côté serveur, dans un build agent managé, et stocke l'image. C'est ce qu'on utilise dans le `setup.ipynb` du projet.

## 9. Azure Container Apps (ACA) — la solution managée d'Azure

ACA est la plateforme **container-as-a-service** d'Azure. C'est **du Kubernetes managé** (literalement : sous le capot, c'est AKS) **avec une couche d'abstraction** par-dessus pour que tu n'aies pas à toucher au YAML k8s, au RBAC k8s, à Helm, etc.

Concrètement : tu donnes une image Docker, Azure se débrouille pour la faire tourner, la scaler, lui filer un HTTPS, faire les rolling updates. Les concepts qu'on a vus en §5 (pod = replica, deployment = revision, service = ingress) sont **toujours là**, simplement renommés et exposés via une API plus simple.

### 6.1 Pourquoi ACA et pas autre chose ?

| Option | Quand ? | Inconvénient |
|--------|---------|--------------|
| **Azure Virtual Machine** | Tu veux du contrôle total OS, kernel custom | Tu gères patchs, scaling, HTTPS, monitoring — trop bas niveau |
| **Azure App Service** | App web simple Python/Node/.NET sans Docker | Pas de Job event-driven, contraint sur runtimes |
| **Azure Functions** | Code court triggé par event, runtime imposé | Tu ne contrôles pas l'image Docker, deps OS limitées |
| **Azure Container Apps** ⭐ | Containers + scaling auto + Jobs event-driven, **sans K8s à gérer** | Quelques limites avancées vs AKS pur |
| **Azure Kubernetes Service (AKS)** | 50+ services, mesh, opérateurs custom | Courbe d'apprentissage énorme, beaucoup à opérer |

**Règle simple** : commence par ACA. Tu passeras à AKS si un jour tu en as **vraiment** besoin (tu ne l'auras probablement jamais).

### 6.2 Hiérarchie des objets ACA — à mémoriser

ACA introduit **5 objets** que tu vas voir partout. Comprends bien la différence, c'est crucial pour lire les Bicep et les logs.

```
  Azure Container Apps Environment           ← 1. L'"env" = ton cluster logique
  (aca-stage-<id>)                              - 1 par projet (ou par env dev/staging/prod)
      │   - réseau VNet partagé (ou public)
      │   - 1 Log Analytics Workspace partagé
      │   - Dapr & service-mesh dispo (option)
      │
      ├── Container App  web                 ← 2. Un service long-running (toujours up)
      │       │
      │       ├── Revision  web--rs9bna8       ← 3. Une "version déployée" (immuable)
      │       │     │
      │       │     ├── Replica  abc-pod-1     ← 4. Un container qui tourne
      │       │     ├── Replica  abc-pod-2          (= une instance, scalable 1→N)
      │       │     └── Replica  abc-pod-3
      │       │
      │       └── Revision  web--prevv7         ← (ancienne revision, peut rester en parallèle)
      │
      └── Container App Job  worker          ← 5. Un Job event-driven (s'éteint quand fini)
              │
              └── Execution  worker-xyz123    ← Un "run" du job (= une revision+replica jetable)
```

**1. Container App Environment** — le "cluster". Tu en crées généralement **un seul par projet/env**. C'est lui qui :
- Partage le réseau (les apps de la même env se voient directement, par DNS interne).
- Centralise les logs vers Log Analytics.
- Offre Dapr (service discovery, pub/sub) en option.

**2. Container App** — ton **service** long-running. Une app = un workload qui doit être constamment up (web, API, frontend). Ingress HTTPS automatique avec un FQDN du genre `web-stage-sadk.gentlesea-80726f00.westeurope.azurecontainerapps.io`.

**3. Revision** — **c'est le concept le plus important d'ACA, et le plus mal compris**. Une revision = un **snapshot immuable** d'une version de ton app : image + variables d'env + CPU/RAM + config scale. **Tu ne modifies jamais une revision** — quand tu changes quoi que ce soit (nouvelle image, env vars, etc.), ACA crée une **nouvelle revision** et bascule le trafic dessus.

C'est ce qui te permet :
- **Rolling updates sans downtime** : la nouvelle revision monte, prend le trafic, l'ancienne s'éteint après.
- **Rollback instantané** : tu ré-actives l'ancienne revision en 1 commande.
- **Blue/green** ou **canary** : tu peux faire cohabiter 2 revisions en "multi-revision mode" et splitter le trafic (ex. 90% v1, 10% v2).

Nommage : `<app-name>--<suffixe-aléatoire>` (ex. `web-stage-sadk--rs9bna8`). Tu peux forcer un suffixe via `revisionSuffix`.

> ⚠️ Par défaut, ACA est en **single-revision mode** (seule la dernière revision reçoit le trafic). Pour faire du A/B testing, il faut passer en **multi-revision mode** dans `properties.configuration.activeRevisionsMode`.

**4. Replica** — un container qui tourne réellement. Une revision peut avoir **1 à N replicas** (config `scale.minReplicas` / `maxReplicas`). C'est juste "combien d'instances Docker du même code". KEDA décide combien lancer (cf. §7).

Chez nous : `web` a `minReplicas: 1` (toujours au moins 1 container up, pour pas avoir de cold start sur la page d'accueil) et `maxReplicas: 3`.

**5. Container App Job & Execution** — contrairement à une App, un **Job ne s'exécute pas en continu**. Il a des **executions** : `worker-stage-sadk-lfrnz` est une execution qui s'est lancée, a traité 1 message, s'est éteinte. Pas de notion de "revision" pour les Jobs — chaque execution prend l'image et la config du job au moment du déclenchement.

Un Job a 3 modes de trigger :
- `Manual` : tu lances toi-même via `az containerapp job start ...`
- `Schedule` : cron (`"0 */6 * * *"` = toutes les 6h)
- `Event` : **KEDA déclenche** (queue, etc.) — c'est notre `worker`

### 6.3 Ingress — comment le trafic entre

**Container App** seulement (les Jobs n'ont pas d'ingress).

```yaml
ingress:
  external: true            # exposer sur Internet (true) ou seulement intra-env (false)
  targetPort: 3000          # port écouté par le container (port HTTP de Next.js chez nous)
  transport: auto           # http / http2 / tcp / auto
  allowInsecure: false      # forcer HTTPS
```

ACA te file gratuitement :
- Un **FQDN HTTPS** avec un certificat géré par Microsoft.
- Un **load balancer** qui répartit le trafic sur tes replicas.
- Le **scaling** : ACA peut spawner un nouveau replica si la charge HTTP monte (scaler `http` natif).

Tu peux brancher ton propre nom de domaine avec un certif TLS apporté ou ManagedCertificate Azure.

### 6.4 Scaling — comment ACA décide combien de replicas/executions

Chaque revision ou job a une config `scale` :

```yaml
scale:
  minReplicas: 1            # plancher (1 = toujours au moins 1 up)
  maxReplicas: 3            # plafond
  rules:
    - name: http-rule
      http:
        metadata:
          concurrentRequests: 50   # ajoute 1 replica si > 50 req simultanées
```

Trois familles de scalers :
- **HTTP scaler** (ACA natif) : pour les Container Apps avec ingress — scale sur req/s.
- **TCP scaler** (ACA natif) : pour les apps TCP non-HTTP.
- **KEDA scalers** (50+ types) : queue, service bus, kafka, cron, prometheus… — cf. §7.

Si `minReplicas: 0` → **scale-to-zero** : 0 € de compute quand inactif (avec cold start de ~5 s au premier hit).

### 6.5 Apps vs Jobs — quand utiliser quoi ?

| | **Container App** | **Container App Job** |
|---|-------------------|-----------------------|
| Modèle | Long-running, **toujours up** (au moins 1 replica si `minReplicas≥1`) | **Exécutions finies** : le job démarre, fait son truc, s'arrête |
| Concept de version | **Revision** (immuable) | (pas de revision, chaque execution prend la config courante) |
| Trigger | HTTP / TCP / event sur replica | **Manual** / **Schedule** (cron) / **Event** (KEDA) |
| Scaling | KEDA + HTTP scaler | KEDA event-driven, ou rien (manual/schedule) |
| Idéal pour | API web, BFF, frontend, gRPC service | Worker queue, batch ETL, cron, traitement async |
| Coût | Replica idle = facturé (si `minReplicas ≥ 1`) | **0 € si rien ne tourne** (parfait pour event-driven) |

**Dans `mailroom-ia`** : `web` = Container App (Next.js sur HTTP), `worker-stage-<id>` = Container App **Job** event-driven (KEDA déclenche une execution par batch de messages dans la queue Storage).


## 10. KEDA — *Kubernetes Event Driven Autoscaler*

**KEDA** est l'autoscaler open-source qui équipe nativement ACA. Tu n'as **rien à installer** — il tourne dans l'env, invisible.

### 7.1 Le problème que KEDA résout

Imagine : tu as une queue avec des messages à traiter. Combien de workers faut-il lancer ?

Approche naïve : un worker qui tourne en boucle, qui poll la queue toutes les X secondes. Ça marche mais :
- Tu payes 24/7 même quand la queue est vide.
- Si la queue gonfle d'un coup (10 000 messages), un seul worker rame.

Approche KEDA : KEDA écoute la queue **à ta place** et dit à ACA "là il faut N workers". Quand la queue est vide → 0 worker. Quand y'en a 100 → 10 workers en parallèle (capé par toi).

### 7.2 Les scalers KEDA — 50+ disponibles

| Source d'événement | Scaler KEDA | Cas d'usage |
|-------------------|-------------|-------------|
| Longueur d'une **Storage Queue** Azure | `azure-queue` | Notre `worker` |
| Messages **Service Bus** | `azure-servicebus` | Workflows entreprise |
| Topics **Event Hubs** | `azure-eventhub` | Streaming haut débit |
| Blobs créés | `azure-blob` | Trigger sur upload de fichier |
| Lag d'un topic **Kafka** | `kafka` | Pipeline data |
| Charge CPU / mémoire | `cpu`, `memory` | Workloads compute-bound |
| **Cron** (calendrier) | `cron` | Tasks périodiques |
| Requêtes HTTP/sec | `http` (Container Apps natif) | Web apps |
| Métrique custom Prometheus | `prometheus` | Tout sur mesure |
| … | … | … |

### 7.3 Exemple complet — notre worker `mailroom-ia`

Tiré du Bicep `infra-apps.bicep` du projet :

```yaml
eventTriggerConfig:
  replicaCompletionCount: 1   # 1 execution = 1 replica qui tourne et se finit
  parallelism: 4              # KEDA peut lancer jusqu'à 4 replicas par execution batch
  scale:
    minExecutions: 0          # scale-to-zero quand rien à faire → 0 €
    maxExecutions: 10         # cap global à 10 executions parallèles
    pollingInterval: 30       # KEDA vérifie la queue toutes les 30 s
    rules:
      - name: queue-length
        type: azure-queue
        metadata:
          accountName: stmailroomXXX
          queueName: doc-to-classify
          queueLength: '1'    # ratio : 1 message → 1 execution déclenchée
        identity: <UAMI-id>   # KEDA s'authentifie via la Managed Identity du Job
```

**Comportement concret étape par étape :**
1. Tu uploades un PDF → `web` pose un message dans la queue.
2. ~15 s plus tard, KEDA réveille (poll toutes les 30 s).
3. KEDA lit la métrique "approximate-messages-count" sur la queue → voit `>= 1`.
4. KEDA dit à ACA "lance 1 execution du job `worker`".
5. ACA démarre un pod (cold start ~5 s), le worker lit le message, fait son boulot, s'éteint.
6. Si entre temps la queue a 5 messages → KEDA déclenche jusqu'à 5 executions en parallèle (mais capé par `maxExecutions: 10`).
7. Queue vide → plus rien ne tourne → **0 € de compute**.

### 7.4 Pourquoi KEDA et pas Azure Functions ?

- **Functions** est ultra simple si tu acceptes ses contraintes : runtime imposé (Python 3.11, Node 22…), pas de Dockerfile custom, pas d'installation de paquets OS (`apt-get install poppler-utils` impossible).
- **ACA Job + KEDA** = tu contrôles entièrement l'image. Notre worker a besoin du SDK `azure-ai-documentintelligence` + `azure-ai-projects` + `openai` + `aiohttp` (empilement Python lourd) → KEDA gagne.

> 🔐 **Important** : KEDA lit la queue via la **Managed Identity** du Job (pas via une connection string avec clé). C'est pour ça que dans `infra-apps.bicep` on assigne les rôles `Storage Queue Data Reader` (pour le poll KEDA) **et** `Storage Queue Data Message Processor` (pour consommer le message côté worker code) à l'identité du worker.

## 10.5 — La vie d'une révision (à lire avant le hands-on)

Quand tu fais `az containerapp update --image new-image:v2`, ACA :
1. Crée une **nouvelle revision** `web--new-suffix` avec ta nouvelle image.
2. Démarre les `minReplicas` nouveaux pods sur cette nouvelle revision.
3. Attend qu'ils soient healthy (readiness probe).
4. **Bascule le trafic** progressivement vers la nouvelle revision.
5. Marque l'ancienne revision comme `inactive` (les pods s'éteignent au bout de quelques minutes).

**Commandes utiles à connaître :**

```bash
# Lister toutes les revisions d'une app
az containerapp revision list -n web-stage-<id> -g <rg> -o table

# Voir laquelle est active
az containerapp show -n web-stage-<id> -g <rg> \
  --query "properties.latestRevisionName" -o tsv

# Rollback : ré-activer une ancienne revision
az containerapp revision activate -n web-stage-<id> -g <rg> \
  --revision web-stage-<id>--ancienne-suffix

# Forcer le redémarrage d'une revision (utile après changement de Managed Identity)
az containerapp revision restart -n web-stage-<id> -g <rg> \
  --revision web-stage-<id>--rs9bna8
```

> ⚠️ **Quand tu changes** : image, env vars, CPU/RAM, scale config, ingress config → **nouvelle revision**.
> Quand tu changes : tags Azure, RBAC, secrets référencés depuis Key Vault → **pas de nouvelle revision** (modif "in-place").

### Lire un log de revision en KQL

Les logs de tes containers (stdout/stderr) atterrissent dans Log Analytics, table `ContainerAppConsoleLogs_CL`. Exemple :

```kusto
ContainerAppConsoleLogs_CL
| where TimeGenerated > ago(15m)
| where ContainerAppName_s == 'web-stage-sadk'
| where RevisionName_s == 'web-stage-sadk--rs9bna8'
| project TimeGenerated, ReplicaName_s, Log_s
| order by TimeGenerated desc
| take 50
```

Tu pourras filtrer par revision spécifique pour diagnostiquer un déploiement qui a mal tourné.

## 11. Hands-on — déployer notre mini container sur ACA

On va :
1. Push l'image `hello-stage:v1` qu'on a buildée en §4 vers un nouvel ACR éphémère
2. Créer un ACA Environment
3. Déployer un Container App qui tourne notre image

👇 **Action requise** : remplacez par votre RG (la même variable que les autres notebooks).

In [ ]:
import os, re, subprocess

RG = "rg-stage-<votre-identifiant>"
m = re.match(r"^rg-stage-(?P<id>[a-z0-9\-]+)$", RG)
assert m, f"❌ Le RG '{RG}' ne suit pas la convention 'rg-stage-<id>'."
USER_ID = m.group("id")

LOCATION = subprocess.check_output(
    f"az group show --name {RG} --query location -o tsv", shell=True
).decode().strip()

clean = USER_ID.replace("-", "")
ACR_NAME = f"acrhello{clean}"
ACA_ENV = f"acaenv-hello-{USER_ID}"
ACA_APP = f"hello-{USER_ID}"

for k, v in {"RG": RG, "USER_ID": USER_ID, "LOCATION": LOCATION,
             "ACR_NAME": ACR_NAME, "ACA_ENV": ACA_ENV, "ACA_APP": ACA_APP}.items():
    os.environ[k] = v

print(f"RG       : {RG}")
print(f"ACR      : {ACR_NAME}")
print(f"ACA env  : {ACA_ENV}")
print(f"ACA app  : {ACA_APP}")

In [ ]:
# Installer l'extension containerapp + enregistrer les providers (idempotent)
!az extension add --name containerapp --upgrade --only-show-errors
!az provider register --namespace Microsoft.App --wait
!az provider register --namespace Microsoft.OperationalInsights --wait

In [ ]:
# 1. Créer un ACR éphémère et y builder l'image (cloud-side, pas besoin de Docker local)
!az acr create --name $ACR_NAME --resource-group $RG --sku Basic --output table
!az acr build --registry $ACR_NAME --image hello-stage:v1 ./_docker-demo

In [ ]:
# 2. Créer l'ACA Environment
!az containerapp env create \
    --name $ACA_ENV \
    --resource-group $RG \
    --location $LOCATION \
    --output table

In [ ]:
# 3. Déployer le Container App
#    On utilise --system-assigned pour activer la Managed Identity
#    et --registry-identity system pour pull via cette identité (pas de mot de passe ACR).
!az containerapp create \
    --name $ACA_APP \
    --resource-group $RG \
    --environment $ACA_ENV \
    --image "$ACR_NAME.azurecr.io/hello-stage:v1" \
    --registry-server "$ACR_NAME.azurecr.io" \
    --system-assigned \
    --registry-identity system \
    --min-replicas 0 --max-replicas 1 \
    --cpu 0.25 --memory 0.5Gi \
    --output table

In [ ]:
# 4. Voir les logs (le container a tourné, affiché Hello, et s'est terminé)
!az containerapp logs show --name $ACA_APP --resource-group $RG --tail 50

🎉 Vous venez de **builder, push et déployer un container sur Azure Container Apps**. C'est exactement le pattern qu'on industrialise dans le projet `mailroom-ia` — en plus gros.

## 12. Pour cleaner ce notebook

On garde l'ACA env + l'ACR créés ici pour le **notebook 09** (on va les réutiliser pour démontrer Bicep). Le cleanup final du parcours arrive dans le 09.

Si vous voulez quand même supprimer maintenant les ressources créées par ce notebook :

In [ ]:
# Voir ce qui existe dans le RG
!az resource list --resource-group $RG --output table

In [ ]:
# Cleanup du Container App seul (optionnel)
# Décommentez si vous voulez juste supprimer l'app, en gardant l'ACA env et l'ACR pour le notebook 08.

# !az containerapp delete --name $ACA_APP --resource-group $RG --yes

print("💡 Astuce : laissez tout en place pour le notebook 09 (Bicep) qui va réutiliser l'ACA env.")

## Récap

Vous savez maintenant :
- ce qu'est un **container** vs une VM ;
- lire et écrire un **Dockerfile** simple (multi-stage, non-root, pin de version) ;
- ce qu'est un **registry** (Docker Hub, MCR, ACR) ;
- la différence **Container App** vs **Container App Job** ;
- ce qu'est **KEDA** et comment ça scale event-driven ;
- déployer un container sur ACA avec Managed Identity et pull ACR sans secret.

### 🚀 Prochaine étape : **notebook 09 — Bicep & Infrastructure as Code**

Vous avez tout déployé jusque-là avec des commandes `az` impératives. Le notebook 09 va vous apprendre à **décrire votre infra en code Bicep** et à la déployer en une commande. C'est le même langage qu'on utilise dans le projet `mailroom-ia` (cf. `projet/mailroom-ia/infra/`).

Le **cleanup final** du parcours sera également dans le notebook 09.

📚 Pour aller plus loin :
- Doc ACA : https://learn.microsoft.com/azure/container-apps/
- Doc KEDA : https://keda.sh/docs/
- Best practices Docker : https://docs.docker.com/develop/develop-images/dockerfile_best-practices/
- ACR : https://learn.microsoft.com/azure/container-registry/